In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import folium
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings("ignore")


from statsmodels.tsa.seasonal import seasonal_decompose
from plotly.subplots import make_subplots


In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/data_dsmarket/'
except ImportError:
    DATA_PATH = 'data_dsmarket/'


df = pd.read_feather(DATA_PATH + 'df_preprocessed.feather')

## Análisis

In [3]:
muestra_producto_tienda = df [ df.id == 'SUPERMARKET_3_594_NYC_2'] # son las ventas de un producto en una tienda solamente
px.line(muestra_producto_tienda, x = 'date', y = 'sales')

In [4]:
muestra_producto = df[df.item == 'SUPERMARKET_3_594'].groupby('date').sales.sum().reset_index()
muestra_producto['date'] = pd.to_datetime(muestra_producto['date'])
fig = px.line(muestra_producto, x='date', y='sales')
eventos = df[df['event'].notna()]
eventos['date'] = pd.to_datetime(eventos['date'])
for fecha in eventos['date'].unique():
    fig.add_vline(x=pd.Timestamp(fecha).timestamp() * 1000, line_dash="dash", line_color="red", opacity=0.5)
fig.show()

In [5]:
# Vamos a ver cuanto vende cada tienda
df.groupby(by = ['store_code','date'])['sales'].sum().reset_index()

,store_code,date,sales
0,BOS_1,2011-01-29,2556
1,BOS_1,2011-01-30,2687
2,BOS_1,2011-01-31,1822
3,BOS_1,2011-02-01,2258
4,BOS_1,2011-02-02,1694
...,...,...,...
19125,PHI_3,2016-04-20,3159
19126,PHI_3,2016-04-21,3226
19127,PHI_3,2016-04-22,3828
19128,PHI_3,2016-04-23,4686


In [6]:
fig = px.line(df.groupby(by = ['store_code','date'])['sales'].sum().reset_index(), x = 'date', y = 'sales', color = 'store_code')
for fecha in eventos['date'].unique():
    fig.add_vline(x=pd.Timestamp(fecha).timestamp() * 1000, line_dash="dash", line_color="red", opacity=0.5)
fig.show()

In [7]:
# dias 25 de diciembre con ventas distintas de 0
df[(df.date.dt.month == 12) & (df.date.dt.day == 25) & (df.sales != 0)]

,id,item,category,department,store,store_code,region,d,sales,date,weekday,event,yearweek,sell_price,season,ingresos,is_holiday
24482452,HOME_&_GARDEN_2_184_PHI_1,HOME_&_GARDEN_2_184,HOME_&_GARDEN,HOME_&_GARDEN_2,Midtown_Village,PHI_1,Philadelphia,d_1792,1,2015-12-25,Friday,Christmas Day,201551.0,2.4625,Invierno,2.4625,1
32373212,SUPERMARKET_1_082_BOS_3,SUPERMARKET_1_082,SUPERMARKET,SUPERMARKET_1,Back_Bay,BOS_3,Boston,d_1427,1,2014-12-25,Thursday,Christmas Day,201451.0,1.1760,Invierno,1.1760,1
32473871,SUPERMARKET_1_087_NYC_3,SUPERMARKET_1_087,SUPERMARKET,SUPERMARKET_1,Tribeca,NYC_3,New York,d_697,1,2012-12-25,Tuesday,Christmas Day,201252.0,6.2760,Invierno,6.2760,1
32641397,SUPERMARKET_1_096_BOS_3,SUPERMARKET_1_096,SUPERMARKET,SUPERMARKET_1,Back_Bay,BOS_3,Boston,d_1792,1,2015-12-25,Friday,Christmas Day,201551.0,7.4160,Invierno,7.4160,1
33119282,SUPERMARKET_1_122_BOS_3,SUPERMARKET_1_122,SUPERMARKET,SUPERMARKET_1,Back_Bay,BOS_3,Boston,d_1427,1,2014-12-25,Thursday,Christmas Day,201451.0,2.3760,Invierno,2.3760,1
37325969,SUPERMARKET_2_125_BOS_2,SUPERMARKET_2_125,SUPERMARKET,SUPERMARKET_2,Roxbury,BOS_2,Boston,d_1427,1,2014-12-25,Thursday,Christmas Day,201451.0,2.6640,Invierno,2.6640,1
44078494,SUPERMARKET_3_080_BOS_2,SUPERMARKET_3_080,SUPERMARKET,SUPERMARKET_3,Roxbury,BOS_2,Boston,d_1062,1,2013-12-25,Wednesday,Christmas Day,201351.0,1.8960,Invierno,1.8960,1
44084598,SUPERMARKET_3_080_NYC_2,SUPERMARKET_3_080,SUPERMARKET,SUPERMARKET_3,Harlem,NYC_2,New York,d_1427,1,2014-12-25,Thursday,Christmas Day,201451.0,2.0160,Invierno,2.0160,1
44090702,SUPERMARKET_3_080_PHI_1,SUPERMARKET_3_080,SUPERMARKET,SUPERMARKET_3,Midtown_Village,PHI_1,Philadelphia,d_1792,1,2015-12-25,Friday,Christmas Day,201551.0,2.0160,Invierno,2.0160,1
44091520,SUPERMARKET_3_080_PHI_2,SUPERMARKET_3_080,SUPERMARKET,SUPERMARKET_3,Yorktown,PHI_2,Philadelphia,d_697,1,2012-12-25,Tuesday,Christmas Day,201252.0,1.8960,Invierno,1.8960,1


In [8]:
fig = px.bar(df.groupby(['category', 'store_code'])['sales'].sum().reset_index(), x='category', y='sales', color='store_code', barmode='stack')
fig.show()

In [9]:
ventas_dia = df.groupby(by = ['store_code', 'weekday'])['sales'].sum().reset_index()
ventas_dia = ventas_dia.pivot(index = 'weekday', columns = 'store_code', values = 'sales')
fig = px.imshow(ventas_dia, color_continuous_scale = 'viridis')
fig.show()

In [10]:
fig = px.box(df.groupby(['store_code', 'date'])['sales'].sum().reset_index(), x='store_code', y='sales', color='store_code')
fig.show()

## PRODUCTOS

### Top productos por ciudad

In [11]:
top_productos = df.groupby('item')['sales'].sum().sort_values(ascending=False).head(10)
print("top productos en general:")
print(top_productos)

for region in ['New York', 'Boston', 'Philadelphia']:
    print(f"\nTop 10 en {region}:")
    top_region = df[df['region'] == region].groupby('item')['sales'].sum().sort_values(ascending=False).head(10)
    print(top_region)

top productos en general:
item
SUPERMARKET_3_090    1002529
SUPERMARKET_3_586     920242
SUPERMARKET_3_252     565299
SUPERMARKET_3_555     491287
SUPERMARKET_3_714     396172
SUPERMARKET_3_587     396119
SUPERMARKET_3_694     390001
SUPERMARKET_3_226     363082
SUPERMARKET_3_202     295689
SUPERMARKET_3_723     284333
Name: sales, dtype: int64

Top 10 en New York:
item
SUPERMARKET_3_090    486138
SUPERMARKET_3_586    318050
SUPERMARKET_3_252    237172
SUPERMARKET_3_120    176446
SUPERMARKET_3_587    166065
SUPERMARKET_3_808    165778
SUPERMARKET_3_635    158166
SUPERMARKET_3_541    156589
SUPERMARKET_3_714    146000
SUPERMARKET_3_555    141146
Name: sales, dtype: int64

Top 10 en Boston:
item
SUPERMARKET_3_586    455411
SUPERMARKET_3_090    328034
SUPERMARKET_3_252    257079
SUPERMARKET_3_555    244685
SUPERMARKET_3_377    164976
SUPERMARKET_3_587    138710
SUPERMARKET_3_714    129772
SUPERMARKET_3_202    123263
SUPERMARKET_3_030    109496
SUPERMARKET_3_694    104541
Name: sales, dtyp

### Productos en declive

In [12]:
# Para eso vamos a agrupar por meses para ver cuales van disminuyendo
df['year_month'] = df['date'].dt.to_period('M')
ventas_mensuales = df.groupby(by = ['item', 'year_month']).sales.sum().reset_index()
ventas_mensuales['year_month'] = ventas_mensuales['year_month'].astype(str)
df

,id,item,category,department,store,store_code,region,d,sales,date,weekday,event,yearweek,sell_price,season,ingresos,is_holiday,year_month
0,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_1,0,2011-01-29,Saturday,NaN,201105.0,12.7414,Invierno,0.0,0,2011-01
1,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_2,0,2011-01-30,Sunday,NaN,201105.0,12.7414,Invierno,0.0,0,2011-01
2,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_3,0,2011-01-31,Monday,NaN,201105.0,12.7414,Invierno,0.0,0,2011-01
3,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_4,0,2011-02-01,Tuesday,NaN,201105.0,12.7414,Invierno,0.0,0,2011-02
4,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_5,0,2011-02-02,Wednesday,NaN,201105.0,12.7414,Invierno,0.0,0,2011-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58327365,SUPERMARKET_3_827_PHI_3,SUPERMARKET_3_827,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1909,0,2016-04-20,Wednesday,NaN,201616.0,1.2000,Primavera,0.0,0,2016-04
58327366,SUPERMARKET_3_827_PHI_3,SUPERMARKET_3_827,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1910,0,2016-04-21,Thursday,NaN,201616.0,1.2000,Primavera,0.0,0,2016-04
58327367,SUPERMARKET_3_827_PHI_3,SUPERMARKET_3_827,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1911,0,2016-04-22,Friday,NaN,201616.0,1.2000,Primavera,0.0,0,2016-04
58327368,SUPERMARKET_3_827_PHI_3,SUPERMARKET_3_827,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1912,0,2016-04-23,Saturday,NaN,201617.0,1.2000,Primavera,0.0,0,2016-04


In [13]:
producto_ejemplo = ventas_mensuales['item'].iloc[120] # este por ejemplo tiene siempre picos en diciembre
data_ejemplo = ventas_mensuales[ventas_mensuales['item'] == producto_ejemplo]
fig = px.line(data_ejemplo, x='year_month', y='sales', title=f'Ventas mensuales: {producto_ejemplo}')
fig.show()

In [14]:
def calcular_tendencia_producto(df):
   # esta función calcula la pendiente de la tendencia de las ventas mensuales de un producto usando descomposición estacional y regresión lineal sobre la tendencia
    try:
        if len(df) < 24:
            return np.nan
        resultado = seasonal_decompose(df['sales'], model = 'multiplicative', period=6)   # CAMBIAR A ADITIVA
        tendencia = resultado.trend.dropna()    # quito el principio y el final pq se me quedan muchos nulos
        x = np.arange(len(tendencia))
        pendiente = np.polyfit(x, tendencia, 1)[0]
        return pendiente
    except:
        return np.nan


tendencias_productos = []
for item in ventas_mensuales['item'].unique():
    producto = ventas_mensuales[ventas_mensuales['item'] == item].sort_values('year_month')
    pendiente = calcular_tendencia_producto(producto)
    tendencias_productos.append({
        'item': item,
        'pendiente_tendencia': pendiente,
        'ventas_totales': producto['sales'].sum()
    })
df_tendencias = pd.DataFrame(tendencias_productos)
df_tendencias = df_tendencias.dropna()
df_tendencias = df_tendencias.sort_values('pendiente_tendencia')
print("productos con pendiente + negativa:")
print(df_tendencias.head(20))

productos con pendiente + negativa:
                   item  pendiente_tendencia  ventas_totales
2374  SUPERMARKET_3_150           -96.357509          147532
2186  SUPERMARKET_2_360           -78.836104          257119
2779  SUPERMARKET_3_555           -70.333456          491287
2810  SUPERMARKET_3_586           -63.211339          920242
2859  SUPERMARKET_3_635           -56.388234          282134
1954  SUPERMARKET_2_128           -55.132040          149465
1848  SUPERMARKET_2_021           -43.940724          121679
2378  SUPERMARKET_3_154           -43.141725          101583
2670  SUPERMARKET_3_446           -41.866947           96285
2692  SUPERMARKET_3_468           -39.319181           85612
2304  SUPERMARKET_3_080           -37.797090          262650
2600  SUPERMARKET_3_376           -36.198017          228551
2102  SUPERMARKET_2_276           -34.874125          158887
3029  SUPERMARKET_3_808           -32.777090          281879
2543  SUPERMARKET_3_319           -29.362151     

In [15]:
productos_declive = df_tendencias.head(10)
fig = make_subplots(rows=5, cols=2, subplot_titles=[f"{prod}" for prod, pend in zip(productos_declive['item'], productos_declive['pendiente_tendencia'])])

for idx, prod in enumerate(productos_declive['item']):
    datos_prod = ventas_mensuales[ventas_mensuales['item'] == prod].sort_values('year_month')
    fig.add_trace(go.Scatter(x=datos_prod['year_month'], y=datos_prod['sales'], mode='lines+markers', name=prod, showlegend=False), row=idx//2+1, col=idx%2+1)

fig.update_layout(height=1200, title_text="Top 10 Productos en Declive")
fig.show()

In [16]:
# vamos a calcular productos que no se hayan vendido en los últimos 2 meses
ultimo_mes = df['date'].max() - pd.DateOffset(months=2)
productos_no_vendidos = df[df['date'] > ultimo_mes].groupby('item')['sales'].sum().reset_index()
productos_no_vendidos = productos_no_vendidos[productos_no_vendidos['sales'] == 0]
print("productos sin ventas en los últimos 2 meses:")
print(productos_no_vendidos)
# Yo aquo nno se si los quitaria directamente, miraria a ver como son sus graficas por ver si es q no se venden en estas fechas del año pero en el resto sí


productos sin ventas en los últimos 2 meses:
                     item  sales
326      ACCESORIES_1_335      0
525      ACCESORIES_2_110      0
1297  HOME_&_GARDEN_2_202      0
1305  HOME_&_GARDEN_2_210      0
1551  HOME_&_GARDEN_2_456      0
1597  HOME_&_GARDEN_2_502      0
1729    SUPERMARKET_1_120      0
1735    SUPERMARKET_1_126      0
2118    SUPERMARKET_2_292      0
2429    SUPERMARKET_3_205      0
2434    SUPERMARKET_3_210      0
2665    SUPERMARKET_3_441      0
2668    SUPERMARKET_3_444      0
2871    SUPERMARKET_3_647      0


### productos con mas diferencia de precios entre tiendas

In [17]:
precios = df.groupby('item')['sell_price'].describe() # tenemos los 3049 productos, vamos a quedarnos con los q presentasen mayor varianza en los precios
precios['rango'] = precios['max'] - precios['min']
precios.sort_values('rango', ascending=False).head(20)

,count,mean,std,min,25%,50%,75%,max,rango
item,,,,,,,,,
HOME_&_GARDEN_2_406,19130.0,16.072735,6.071041,4.0750,15.5750,15.5750,15.5875,134.1500,130.0750
HOME_&_GARDEN_2_466,19130.0,9.410104,5.427666,7.4625,8.7125,8.7125,8.7125,65.7750,58.3125
HOME_&_GARDEN_2_178,19130.0,8.742881,1.558619,3.7500,8.7125,8.7125,8.7125,55.4500,51.7000
HOME_&_GARDEN_2_250,19130.0,8.797284,1.374992,4.2000,8.7000,8.7000,8.7125,42.7250,38.5250
HOME_&_GARDEN_2_514,19130.0,23.468817,1.153093,1.2500,22.4250,23.7125,23.7125,26.2125,24.9625
ACCESORIES_1_342,19130.0,26.162844,1.145219,2.6600,25.2301,26.4404,27.1054,27.1054,24.4454
HOME_&_GARDEN_1_469,19130.0,22.467594,0.876563,1.2500,22.4625,22.4625,22.4625,25.5750,24.3250
ACCESORIES_1_257,19130.0,22.344449,0.507519,10.5868,22.2908,22.2908,22.2908,34.3140,23.7272
HOME_&_GARDEN_1_342,19130.0,22.425953,0.590805,1.2500,22.4625,22.4625,22.4625,22.4625,21.2125


In [18]:
precios.sort_values(by = 'std', ascending = False).head(10)

,count,mean,std,min,25%,50%,75%,max,rango
item,,,,,,,,,
HOME_&_GARDEN_2_406,19130.0,16.072735,6.071041,4.0750,15.5750,15.5750,15.5875,134.1500,130.0750
HOME_&_GARDEN_2_466,19130.0,9.410104,5.427666,7.4625,8.7125,8.7125,8.7125,65.7750,58.3125
HOME_&_GARDEN_2_292,19130.0,15.824824,3.149457,12.4625,12.4625,18.7125,18.7125,19.9625,7.5000
HOME_&_GARDEN_1_137,19130.0,15.564437,2.317600,7.4125,16.2250,16.2250,16.2250,16.2250,8.8125
HOME_&_GARDEN_1_259,19130.0,12.143478,2.010412,7.4875,9.3500,13.7125,13.7125,14.4000,6.9125
ACCESORIES_1_311,19130.0,20.880102,1.964060,17.2900,19.4978,19.4978,23.6474,23.9134,6.6234
ACCESORIES_1_354,19130.0,30.680201,1.935434,26.5734,30.5634,31.8934,31.8934,31.8934,5.3200
ACCESORIES_1_158,19130.0,28.691754,1.896873,26.5734,26.5734,29.2201,30.5634,30.5634,3.9900
HOME_&_GARDEN_2_446,19130.0,31.813491,1.863452,18.7375,31.1500,32.4625,32.4625,35.0000,16.2625


1. Rango de precios (max - min): mide la diferencia entre la tienda más cara y la más barata para cada producto. Un rango alto indica que el mismo producto se vende a precios muy diferentes según la ubicación.
2. Desviación estándar: mide cuánto varían los precios en general. Mientras que el rango solo captura los extremos, la desviación estándar considera todos los precios y detecta si la variación es generalizada o si es solo una tienda atípica.

El rango solo mira extremos y la desviación penaliza más cuando MUCHOS precios están dispersos. Yo le preguntaría a Dani porque no sé cual nos puede interesar más

### productos estacionales

In [19]:
ventas_estacion = df.groupby(['item', 'season'])['sales'].sum().reset_index()
ventas_estacion['total_producto'] = ventas_estacion.groupby('item')['sales'].transform('sum')
ventas_estacion['pct'] = (ventas_estacion['sales'] / ventas_estacion['total_producto'] * 100).round(1)
# ventas_estacion
resumen_estaciones = ventas_estacion.pivot(index='item', columns='season', values='pct')
resumen_estaciones['max_season'] = resumen_estaciones.max(axis=1)
productos_estacionales = resumen_estaciones.sort_values('max_season', ascending=False).head(20)
print("Productos más estacionales (concentrados en una época):")
print(productos_estacionales)

Productos más estacionales (concentrados en una época):
season               Invierno  Otoño  Primavera  Verano  max_season
item                                                               
HOME_&_GARDEN_1_297       4.5   13.3       20.1    62.1        62.1
HOME_&_GARDEN_2_162       0.3   23.9       16.0    59.8        59.8
HOME_&_GARDEN_2_340       2.8    5.1       32.6    59.6        59.6
HOME_&_GARDEN_1_189       4.0   18.4       19.6    58.0        58.0
HOME_&_GARDEN_2_022      11.0    6.4       57.9    24.6        57.9
HOME_&_GARDEN_1_049       0.1    3.8       38.5    57.6        57.6
HOME_&_GARDEN_2_449       4.3    9.7       28.4    57.6        57.6
HOME_&_GARDEN_2_289       2.7   15.9       24.0    57.4        57.4
HOME_&_GARDEN_2_344       6.1   17.8       19.7    56.4        56.4
HOME_&_GARDEN_2_080       6.3    7.8       29.8    56.1        56.1
HOME_&_GARDEN_2_477       4.6   19.7       21.1    54.6        54.6
SUPERMARKET_3_350        14.7   13.6       54.1    17.5     

## Ubicación

In [20]:
# vamos a añadir las coordenadas de cada tienda para poder representarlas en un mapa y ver si hay alguna relación entre la ubicación y las ventas
df[['store', 'store_code', 'region']].drop_duplicates().sort_values('store_code')

,store,store_code,region
0,South_End,BOS_1,Boston
1913,Roxbury,BOS_2,Boston
3826,Back_Bay,BOS_3,Boston
5739,Greenwich_Village,NYC_1,New York
7652,Harlem,NYC_2,New York
9565,Tribeca,NYC_3,New York
11478,Brooklyn,NYC_4,New York
13391,Midtown_Village,PHI_1,Philadelphia
15304,Yorktown,PHI_2,Philadelphia
17217,Queen_Village,PHI_3,Philadelphia


In [21]:
coordenadas = {
    'BOS_1': {'store': 'South_End',        'lat': 42.3467, 'lon': -71.0792},
    'BOS_2': {'store': 'Roxbury',          'lat': 42.3188, 'lon': -71.0846},
    'BOS_3': {'store': 'Back_Bay',         'lat': 42.3503, 'lon': -71.0810},
    'NYC_1': {'store': 'Greenwich_Village','lat': 40.7335, 'lon': -74.0027},
    'NYC_2': {'store': 'Harlem',           'lat': 40.8116, 'lon': -73.9465},
    'NYC_3': {'store': 'Tribeca',          'lat': 40.7163, 'lon': -74.0086},
    'NYC_4': {'store': 'Brooklyn',         'lat': 40.6782, 'lon': -73.9442},
    'PHI_1': {'store': 'Midtown_Village',  'lat': 39.9496, 'lon': -75.1609},
    'PHI_2': {'store': 'Yorktown',         'lat': 39.9706, 'lon': -75.1579},
    'PHI_3': {'store': 'Queen_Village',    'lat': 39.9357, 'lon': -75.1527},
}

In [22]:
ventas_tienda = df.groupby('store_code')['sales'].sum().reset_index()
ventas_tienda['lat'] = ventas_tienda['store_code'].map({k: v['lat'] for k, v in coordenadas.items()})
ventas_tienda['lon'] = ventas_tienda['store_code'].map({k: v['lon'] for k, v in coordenadas.items()})
ventas_tienda['store'] = ventas_tienda['store_code'].map({k: v['store'] for k, v in coordenadas.items()})
ventas_tienda['radio'] = (ventas_tienda['sales'] / ventas_tienda['sales'].max()) * 50

colores = ['red', 'blue',   'green', 'orange', 'purple', 'black',  'blue',  'gray',  'pink',  'cadetblue']
colores = ['red', 'darkred', 'orange', 'green', 'darkgreen', 'blue', 'purple', 'darkpurple', 'cadetblue', 'pink']

display(ventas_tienda)


mapa = folium.Map(location=[41.5, -73.5], zoom_start=7,       # mapa centrado en el noreste de EEUU
                  tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
                  attr='Esri World Street Map')
contador = 0
for _, row in ventas_tienda.iterrows():
    colorpunto = colores[contador]
    # print(colorpunto)
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=row['radio'],
        popup=f"{row['store']} ({row['store_code']})<br>Ventas: {row['sales']:,.0f}",
        color = colorpunto,
        fill=True,
        fill_opacity=0.6
    ).add_to(mapa)
    contador += 1

mapa

,store_code,sales,lat,lon,store,radio
0,BOS_1,5595292,42.3467,-71.0792,South_End,25.005372
1,BOS_2,7214384,42.3188,-71.0846,Roxbury,32.241097
2,BOS_3,6089330,42.3503,-71.0810,Back_Bay,27.213229
3,NYC_1,7698216,40.7335,-74.0027,Greenwich_Village,34.403344
4,NYC_2,5685475,40.8116,-73.9465,Harlem,25.408400
5,NYC_3,11188180,40.7163,-74.0086,Tribeca,50.000000
6,NYC_4,4103676,40.6782,-73.9442,Brooklyn,18.339337
7,PHI_1,5149062,39.9496,-75.1609,Midtown_Village,23.011169
8,PHI_2,6544012,39.9706,-75.1579,Yorktown,29.245203
9,PHI_3,6427782,39.9357,-75.1527,Queen_Village,28.725771


## Columnas auxiliares

In [23]:
df['date'] = pd.to_datetime(df['date'])
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month
df['week']  = df['date'].dt.isocalendar().week.astype(int)
df['quarter'] = df['date'].dt.quarter

print(df[['date','year','month','week','sales','ingresos','is_holiday']].head())

        date  year  month  week  sales  ingresos  is_holiday
0 2011-01-29  2011      1     4      0       0.0           0
1 2011-01-30  2011      1     4      0       0.0           0
2 2011-01-31  2011      1     5      0       0.0           0
3 2011-02-01  2011      2     5      0       0.0           0
4 2011-02-02  2011      2     5      0       0.0           0


## Análisis temporal (aunq se podria meter en el utlimo punto de series temporales)

### Heatmap semana × año

In [24]:
pivot = df.groupby(['year','week'])['ingresos'].sum().reset_index().pivot(index='year', columns='week', values='ingresos')
fig = px.imshow(pivot, labels=dict(x='Semana', y='Año', color='Ingresos'), title='Ingresos por semana y año', color_continuous_scale='Viridis')
fig.show()


# ESTE GRAFICO MEJOR UNA SERIE TEMPORAL

In [25]:
#SERIE TEMPORAL DE LO ANTERIOR
serie = df.groupby('date')['ingresos'].sum()
fig = px.line(serie.reset_index(), x='date', y='ingresos',
              title='Ingresos semanales 2011–2015',
              labels={'date': 'Fecha', 'ingresos': 'Ingresos'})
fig.show()

In [ ]:
#Tendencia creciente a lo largo de los años, de ~80k en 2011 a ~160-200k en 2016
#Caídas puntuales a cero que son probablemente días festivos o cierres donde no hay ventas registradas

### Ventas por mes (2011–2015)

In [26]:
nombres_mes = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
media_mes = (df[df['year'] != 2016]
               .groupby('month')['ingresos'].sum().reset_index())
media_mes['mes_nombre'] = media_mes['month'].apply(lambda m: nombres_mes[m-1])

fig = px.bar(media_mes, x='mes_nombre', y='ingresos', title='Ingresos totales x mes',labels={'mes_nombre':'Mes','ingresos':'Ingresos'})
fig.show()

### Ventas por trimestre

In [ ]:
# Q1: enero, febrero, marzo
# Q2: abril, mayo, junio
# Q3: julio, agosto, septiembre
# Q4: octubre, noviembre, diciembre

In [28]:
ventas_q = (df[df['year'] != 2016]
              .groupby(['year','quarter'])['ingresos'].mean().reset_index())
ventas_q['label'] = ventas_q['year'].astype(str) + ' Q' + ventas_q['quarter'].astype(str)

fig = px.bar(ventas_q, x='label', y='ingresos', color='quarter',
             title='Ingresos por trimestre (2011–2015)',
             labels={'label':'Trimestre','ingresos':'Ingresos'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

### Evolución anual

In [27]:
fig = px.bar(df.groupby('year')['ingresos'].sum().reset_index(),
             x='year', y='ingresos',
             title='Ingresos totales por año',
             labels={'year':'Año','ingresos':'Ingresos'})
fig.show()

# HAY Q TENER EN CUENTA Q 2016 SOLO TIENE DATOS DE ENERO Y FEBRERO, POR ESO LO HE EXCLUIDO DE LOS GRÁFICOS ANTERIORES PARA NO DISTORSIONAR LAS TENDENCIAS ANUALES Y MENSUALES.
# SI LO INCLUYO, 2016 APARECE CON INGRESOS MUY BAJOS Y PUEDE DAR UNA FALSA IMPRESIÓN DE QUE HAY UNA CAÍDA EN LOS INGRESOS CUANDO EN REALIDAD ES SOLO PORQUE NO TENEMOS DATOS DE TODO EL AÑO.

## Matriz de correlación

In [28]:
num_cols = [c for c in ['sales', 'sell_price', 'ingresos', 'is_holiday', 'week', 'month', 'year'] if c in df.columns]
corr = df[num_cols].corr().round(2)
fig = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu', color_continuous_midpoint=0, title='Correlación entre variables numéricas', width=700, height=600)
fig.show()

## Análisis de festivos

### ¿Se vende más en días festivos?

In [29]:
"""
QAL CORRER ESTO SE ME MUERE EL KERJNELL
fig = px.box(df, x='is_holiday', y='ingresos', color='is_holiday',
             category_orders={'is_holiday':[0,1]},
             title='Distribución de ingresos: días festivos vs no festivos',
             labels={'is_holiday':'Festivo','ingresos':'Ingresos'})
fig.update_xaxes(tickvals=[0,1], ticktext=['No festivo','Festivo'])
fig.show()"""

"\nQAL CORRER ESTO SE ME MUERE EL KERJNELL\nfig = px.box(df, x='is_holiday', y='ingresos', color='is_holiday',\n             category_orders={'is_holiday':[0,1]},\n             title='Distribución de ingresos: días festivos vs no festivos',\n             labels={'is_holiday':'Festivo','ingresos':'Ingresos'})\nfig.update_xaxes(tickvals=[0,1], ticktext=['No festivo','Festivo'])\nfig.show()"

In [30]:
"""fig = px.box(df.assign(is_holiday=df['is_holiday'].map({0: 'No festivo', 1: 'Festivo'})),
             x='is_holiday', y='ingresos', color='is_holiday',
             category_orders={'is_holiday': ['No festivo', 'Festivo']},
             title='Distribución de ingresos: días festivos vs no festivos',
             labels={'is_holiday': 'Festivo', 'ingresos': 'Ingresos'})
fig.show()
"""
# ESTA CELDA NO ME TGERMINA

"fig = px.box(df.assign(is_holiday=df['is_holiday'].map({0: 'No festivo', 1: 'Festivo'})),\n             x='is_holiday', y='ingresos', color='is_holiday',\n             category_orders={'is_holiday': ['No festivo', 'Festivo']},\n             title='Distribución de ingresos: días festivos vs no festivos',\n             labels={'is_holiday': 'Festivo', 'ingresos': 'Ingresos'})\nfig.show()\n"

In [31]:
media_hol = df.groupby('is_holiday')['ingresos'].mean().reset_index()
media_hol['tipo'] = media_hol['is_holiday'].map({0:'No festivo', 1:'Festivo'})

fig = px.bar(media_hol, x='tipo', y='ingresos', color='tipo',
             title='Media de ingresos: festivo vs no festivo',
             labels={'tipo':'','ingresos':'Ingresos medios'})
fig.show()
# creo q esto no esta bine hehco, no se si es q hay pocos días festivos o q pero no me sale nada, lo dejo comentado por si quiero volver a intentarlo más adelante

### Ingresos en festivos por categoría

In [32]:
hol_cat = df.groupby(['category','is_holiday'])['ingresos'].mean().reset_index()
hol_cat['tipo'] = hol_cat['is_holiday'].map({0:'No festivo', 1:'Festivo'})

fig = px.bar(hol_cat, x='category', y='ingresos', color='tipo', barmode='group',
             title='Ingreso medio por categoría: festivo vs no festivo',
             labels={'category':'Categoría','ingresos':'Ingresos medios','tipo':''})
fig.show()

### ¿Qué eventos generan más ventas?

In [33]:
eventos_df = df[df['event'].notna()]
top_eventos = (eventos_df.groupby('event')['ingresos'].sum()
                         .sort_values(ascending=False).head(15).reset_index())

fig = px.bar(top_eventos, x='ingresos', y='event', orientation='h',
             title='Top 15 eventos por ingresos generados',
             labels={'ingresos':'Ingresos','event':'Evento'})
fig.show()

## Top / Bottom productos

### Top 10 y Bottom 10 globales

In [34]:
ventas_item = df.groupby('item')['ingresos'].sum().reset_index().sort_values('ingresos', ascending=False)
top10    = ventas_item.head(10)
bottom10 = ventas_item.tail(10).sort_values('ingresos')

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Top 10 productos', 'Bottom 10 productos'])
fig.add_trace(go.Bar(x=top10['ingresos'], y=top10['item'], orientation='h',
                     marker_color='steelblue', name='Top 10'), row=1, col=1)
fig.add_trace(go.Bar(x=bottom10['ingresos'], y=bottom10['item'], orientation='h',
                     marker_color='salmon', name='Bottom 10'), row=1, col=2)
fig.update_layout(title='Ingresos totales por producto', height=450, showlegend=False)
fig.show()

### Top 10 por región

In [35]:
for region in sorted(df['region'].unique()):
    top_r = (df[df['region'] == region]
               .groupby('item')['ingresos'].sum()
               .nlargest(10).reset_index().sort_values('ingresos'))
    fig = px.bar(top_r, x='ingresos', y='item', orientation='h',
                 title=f'Top 10 productos — {region}',
                 labels={'ingresos':'Ingresos','item':'Producto'})
    fig.show()

### Top 10 por categoría

In [36]:
for cat in sorted(df['category'].unique()):
    top_c = (df[df['category'] == cat]
               .groupby('item')['ingresos'].sum()
               .nlargest(10).reset_index().sort_values('ingresos'))
    fig = px.bar(top_c, x='ingresos', y='item', orientation='h',
                 title=f'Top 10 productos — {cat}',
                 labels={'ingresos':'Ingresos','item':'Producto'})
    fig.show()


### Top 10 por departamento

In [37]:
for dept in sorted(df['department'].unique()):
    top_d = (df[df['department'] == dept]
               .groupby('item')['ingresos'].sum()
               .nlargest(10).reset_index().sort_values('ingresos'))
    if top_d.empty: continue
    fig = px.bar(top_d, x='ingresos', y='item', orientation='h',
                 title=f'Top 10 productos — {dept}',
                 labels={'ingresos':'Ingresos','item':'Producto'})
    fig.show()